In [27]:
# !pip install torchinfo
# !pip install torch torchvision torchaudio
# mac os version 14.6.1 (23G93)
# !conda install pytorch torchvision torchaudio -c pytorch -y
# !pip uninstall torchvision -y
# !conda uninstall torchvision -y
# !conda install -c pytorch torchvision -y
# !pip install torchvision
# !pip install kaggel --upgrade
!kaggle datasets download -d s076923/pytorch-transformer
import numpy as np, pandas as pd
import sys ;sys.path.append("../../개발자를위한머신러닝/")
from CustomFunc import *

zsh:1: command not found: kaggle


링크 : https://github.com/wikibook/pytorchtrf
데이터세트 : https://www.kaggle.com/datasets/s076923/pytorch-transformer



# Chapter 3. computer vision
# 08. Image classification 
- 객체, 장면 과 같은 요소를 인식하고 분류하는 알고리즘
- 단일클래스 준류(참거짓), 다중클래스 분류(여러클래스), 다중 레이블 분류(한이미지에서 여러클래스)
## AlexNet
- 인식 오류율 26->16% , Lenet-5 => AlexNet, 입력이미지크기, 활성화 함수, 풀링방식의 변경, 드롭아웃 추가 된점이 다름. ReLU 로 기울기 소실 문제를 해결 계층을 깊게 축적가능.
- maxpool 을 이용하면 평균값보다 분포가 더 일정한 효과를 가짐. => 가중치 계산이 더 쉬워짐.
- 드롭아웃으로 과대적함 문제를 해결해서 두배 많은 반복 학습을 수행해 성능을 향상.

- 알렉스넷 모델 불러오기 함수로 대규모 데이터 세트에서 사전학습된 가중치의 알렉스넷을 불러옴.
- AlexNet_Weigths.IMAGENET1k_V1 모델 정보 
    - acc@1(56.522), acc@5(79) : 5개의 레이블에 대한 예측 정확도 79%,
    - 입력이미지 최소크기 63*63 매개변수#(61,100,840)
    - 카테고리수 (1000), 파일긐기 233Mb
- GFLOPS(초당기가 부동 소수점 연산 ,Giga-Floating Point Operations Per Second):
해당 모델에 대한 컴퓨팅 성능을 예측
- rgb 256*256 사용. 
- 예제 8.3 전처리과정 정규화 클래스 Normalize 의 평균과 표준편차 지정(이미지넷 데이터 세트에 사용된 이미지들의 평균과 표준 편차값임. ) 
- 이미지넷 (1400만개 이상의 이미지 포함. 가장 우수한 정규화 값으로 수행됨)
- 알렉스 넷 <-이미지넷 데이터로 학습. Otherwise, 위성사진, 의료사진등 구성이미지는 해당 데이터세트에 적합한 통계치를 계산하여 적용필요
[0.485,0.456,0.406] , [0.229,0.224,0.225]
- (1,3,224,224) 크기의 텐서 입력 => 특징맵의 크기와 필요 매개변수 수 , Layer, Output shape, Parm, 


## 8.3 다중레이블 분류 
- 임의의 값으로 모델을 확인 -> torch.no_grad 클래스로 기울기 계산을 비활성
- 모델에 입력텐서를 전달해 순전파 수행, 반환 출력값 = [배치크기, 클래스개수]
- 전달된 텐서를 소프트맥스 함수로 값을 활성화, topk 메서드로 텐서값이 가장 큰 상위 다섯개 요소만 반환한다. top_idx는 top_probs가 해당하는 색인 값을 의미 
- 확를, 색인, 클래스를 넘파이로 변환 


In [37]:


g("3.8.0 torch , cuda system check and model download")
import torch
print(torch.__version__)
print(torch.backends.mps.is_built())
print(torch.backends.mps.is_available())
print(torch.cuda.is_available())
# device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
g("3.8.0 apple silicon user => Metal Performance Shaders 가속을 사용함. ")
device = "mps" if torch.backends.mps.is_available() and torch.backends.mps.is_built() else "cpu"
print(f"Using {device} device")
tensor = torch.FloatTensor([1,2,3]).to(device)
g("3.8.0pytorch 일부연선 mps 가속 불가능 발생경우 지원안되는 경우도 돌아게게끔 환경 설정")
import os
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] ="1"


g("3.8.1 알렉스넷 모델 학습 ")
from torchvision import models
from torchinfo import summary
model = models.alexnet(weights = "AlexNet_Weights.IMAGENET1K_V1")
summary(model, (1,3,224,224), device ="cpu")

with open("./Datasets/imagenet_classes.txt") as file:
    classes = file.read().splitlines()
    
print(f" AlexNet 클래스 개수: {len(classes)}")
print(f" AlexNet 첫번째 클래스 레이블: {classes[0]}")

g("3.8.3 이미지 데이터 전처리")
from PIL import Image
g(" - torchvision => models, transforms import")
from torchvision import models, transforms

transform = transforms.Compose(
    [
        transforms.Resize((224,224)),
        transforms.ToTensor(),
        transforms.Normalize(
            mean = [0.485, 0.456, 0.406],
            std = [0.229,0.224,0.225]
        ),
    ]
)
print(green(" - transform info"),yellow(transform))


model = models.alexnet(weights="AlexNet_Weights.IMAGENET1K_V1").eval().to(device)
tensors = []
files = ['./datasets/images/airplane.jpg','./datasets/images/bus.jpg']
for file in files:
    image = Image.open(file)
    tensors.append(transform(image))

tensors = torch.stack(tensors)
print(f"입력 텐서의 크기 :{tensors.shape}")

g("3.8.4 알렉스넷 모델 추론")

import numpy as np
from torch.nn import functional as F

with torch.no_grad(): 
    outputs = model(tensors.to(device)) # 반환 출력값
    print(" - model(tensors)output",outputs) # [배치크기, 클래스 개수]
    probs = F.softmax(outputs, dim =-1)
    g(" - probs")
    print(probs)
    top_probs, top_idxs = probs.topk(5)

g(" top5 prob -> probs.topk(5)를 numpy 로 변환 ")
top_probs = top_probs.detach().cpu().numpy()
top_idxs = top_idxs.detach().cpu().numpy()

g(" top classes :")
top_classes = np.array(classes)[top_idxs]
print(top_classes)
for idx, (cls,prob) in enumerate(zip(top_classes, top_probs)):
    print(f"{files[idx]} 추론 결과")
    for classname,probability, in zip(cls,prob):
        print(f" - {classname:<30} : {probability * 100:>5.2f}%")

3.8.0 torch , cuda system check and model download
2.5.1
True
True
False
3.8.0 apple silicon user => Metal Performance Shaders 가속을 사용함. 
Using mps device
3.8.0pytorch 일부연선 mps 가속 불가능 발생경우 지원안되는 경우도 돌아게게끔 환경 설정
3.8.1 알렉스넷 모델 학습 
 AlexNet 클래스 개수: 1000
 AlexNet 첫번째 클래스 레이블: tench
3.8.3 이미지 데이터 전처리
 - torchvision => models, transforms import
 - transform info Compose(
    Resize(size=(224, 224), interpolation=bilinear, max_size=None, antialias=True)
    ToTensor()
    Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
)
입력 텐서의 크기 :torch.Size([2, 3, 224, 224])
3.8.4 알렉스넷 모델 추론
 - model(tensors)output tensor([[-0.2584, -3.9698,  2.8629,  ..., -1.9249,  4.7732, -1.4441],
        [-4.0288, -0.1984,  2.1301,  ..., -5.3826, -2.6092,  1.2260]],
       device='mps:0')
 - probs
tensor([[4.8324e-10, 1.1812e-11, 1.0958e-08,  ..., 9.1287e-11, 7.4024e-08,
         1.4765e-10],
        [3.4604e-13, 1.5946e-11, 1.6365e-10,  ..., 8.9371e-14, 1.4311e-12,
         6.6264e-11]], device='mps:0')

## 3.10 비전 트랜스 포머 
- ViT(Vsion Transformer) an image is Worth 16x16 Words: Transformers for Image Recognition at Scale 논문에서 소개된 이미지 인식을 위한 딥러닝 모델 . 
- 이미지를 자연어 처리 방식 처럼 분류 
- 셀프 어텐션을 적용함. 합성곱 모델은 지역특징 추출이지만 Vit 는 셀프 어텐션으로 전체 이미지를 한번에 처리함. 
- vit 모델은 이미지 패치가 왼쪽에서 오른쪽 위에서 아래방향으로 순차 입력됨. 2차원 구조 이미지 특성 반영안됨 => Swin transformer, Convolutional Vision Transformer(CvT) 제안됨 

- 스윈 트랜스포머 : 로컬윈도를 활용하여 각 계층의 어텐션이 되는 패치의 크기와 개수를 다양하게 구성해 이미지 특성 학습함. 
- Vit 구조와 스윈비교
    - 기존 셀프 어텐션 => 로컬윈도안에 대한 어텐션, 로컬윈도간에 어텐션 으로 수행 => 계층적이미지 학습
    - 어텐션 함수에 상대적 위치 편향을 반영하여 어텐션 값 자체에 위치적 정보 포함 

- CvT 와 비교 
    - 기존 합성곱 연산과정을 ViT 에 적용한 모델 
    - 저수즌 특징(눈,코,입), 고수준 특징(전체얼굴) 을 계층 적으로 반영
    - 어텐션 연산 과정에서 쿼리(Q),키(key,K) , 값(V) 중 키와 값을 기존 특징벡터보다 축소해 계산 복잡도를 감소시킴 
-Vit 계열의 모델은 이미지 분류작업에 매우 효과적으로 입증됨, 광범위한 다른 컴퓨터 비전작업에 적용가능 
- Fashion MNIST 로 실습. 
### 3.10.1 ViT
- ViT 모델구조는 BERT 구조와 다르게 2D 이미지를 1차원 으로 만들어서 CLS 에 입력하는 구조이다.
### 3.10.1 합성곱 모델과 ViT 모델 비교
- 합성곱 신경망은 이미지 특징춫출하여 일부선택 학습, ViT 임베딩은 이미지를 작은 패치들로 나눠 각 패치 간의 상관관계를 학습함. 
- 셀프 어텐션 방법을 사용해 모든 이미지 패치가 서로에게 주는 여향을 고려 => 이미지 전체 특징 추출
- 모든 이미지 패치가 학습에 관여하여 높은 수준의 이미지 표현을 제공함. 
- 좁은수용영역(Receptive Field) 를 가진 ConvNN 은 전체 이미지 정보를 표현하는데 수많은 계층 필요 , 트랜스포머 모델은 어텐션거리를 계산하여 오직 한개의 ViT레이어로 전체 이미지정보를 쉽게 표현가능 . 패치단위 이미지 처리 -> 더 작은 모델로 높은 성능 얻음. 
- Vit 모델은 이미지 크기가 고정됨(전처리필) , Conv 는 공간적위치정보 고려, Vit 는 패치간의 상대적인 위치정보만 고려 => 이미지 변환취약함. 
### 3.10.2 ViT의 귀납적 편향 
- Inductive Bias(귀납적 편향 ): 일반화 성능 향상을 위한 모델의 가정(Assumption)을 의미함. 
- ex) 지역적편향(local bias)을 가진 Cov모델은 이미지데이터의 공간적 관계를 잘 표현. 
- ex1) 시계열 데이터는 시간적 관계를 잘표현 = Sequential편향을 가진 순환신경만 모델
- 순환신경망은 이전 시간의 상태를 기억하고 현재 입력과 결합하여 다음 상태를 예측. 
(24)






Using cpu device


In [ ]:

import torchvision
batch_size = 64

trainset = torchvision.datasets.CIFAR10(root='./data', train=True,
                                        download=True, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=batch_size,
                                        shuffle=True, num_workers=2)

# testset = torchvision.datasets.CIFAR10(root='./data', train=False,
#                                        download=True, transform=transform)
# testloader = torch.utils.data.DataLoader(testset, batch_size=batch_size,
#                                         shuffle=False, num_workers=2)

classes = ('plane', 'car', 'bird', 'cat',
        'deer', 'dog', 'frog', 'horse', 'ship', 'truck')
BATCH_SIZE = 64
num_workers = 4

import matplotlib.pyplot as plt

xb, yb = next(iter(trainloader))

# print(xb)
# row_num, col_num = int(BATCH_SIZE/8), 8
# fig, axs = plt.subplots(row_num, col_num, figsize=(10, 10 * row_num/col_num))

# for row_idx in range(row_num):
#     for col_idx in range(col_num):
#         ax = axs[row_idx][col_idx]
#         i = col_idx * row_num + row_idx
#         class_index = yb[i].item()
#         class_label = classes[class_index].split(",")[0]
#         img = xb[i].permute(1,2,0)
#         ax.title.set_text(class_label)
#         ax.set_yticks([])
#         ax.set_xticks([])
#         ax.imshow(img)
# plt.tight_layout(pad=0.5)
# plt.show()

